### Env and LLM initialisation

In [54]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
import os

load = load_dotenv('./../.env', override=True)


# ollama_cloud_llm = ChatOllama(
#     base_url="http://localhost:11434/",  # Ollama cloud endpoint
#     model="devstral-small-2:24b-cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
#     temperature=0.5,
#     max_tokens=1000,
#     headers={
#         "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
#     }
# )

ollama_local_llm = ChatOllama(
    base_url="http://localhost:11434/",
    model="llama3.2:latest",
    temperature=0.5,
    max_tokens=20,
    num_gpu=999
)

### Using a community (tool)

In [99]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipidia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

tool_response = wikipidia.invoke("Give me details of Avatar Movie Franchise")

print(tool_response)

Page: Avatar (franchise)
Summary: Avatar is an American epic science fiction media franchise created by James Cameron, which began with the 2009 film Avatar. Produced by Lightstorm Entertainment and distributed by 20th Century Studios, it consists of associated merchandise, video games, and theme park attractions.



Page: Avatar: Fire and Ash
Summary: Avatar: Fire and Ash is a 2025 American epic science fiction film directed by James Cameron from a screenplay he co-wrote with Rick Jaffa and Amanda Silver. Produced by Lightstorm Entertainment, it is the third installment in the Avatar film series and the sequel to Avatar: The Way of Water (2022). The film features Sam Worthington, Zoe Saldaña, Sigourney Weaver, Stephen Lang, and Kate Winslet reprising their roles from previous films. The story follows the human-turned-Na'vi Jake Sully and his family on the habitable moon Pandora, as they face the combined threat of the human RDA forces and the Mangkwan, a ruthless Na'vi clan.
Following

### Install DuckduckGo search tool

In [ ]:
#pip install duckduckgo-search langchain-community langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [96]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()
print(search_tool.invoke("LWhat's the latest version of playwright right now?"))

Playwright now supports Ubuntu 20.04 ARM64. You can now run Playwright tests inside Docker on Apple M1 and on Raspberry Pi.This version of Playwright was also tested against the following stable channels: Google Chrome 94. Microsoft Edge 94. Playwright is a framework for Web Testing and Automation. It allows testing Chromium, Firefox and WebKit with a single API. - Releases · microsoft/playwright. Latest version: 1.59.1, last published: a month ago. Start using playwright in your project by running `npm i playwright`. What is Playwright.This command will display the version of npm installed on your machine. If you don’t see a version number, it means npm is not set up on your machine. In that case, you can download and install npm from the official website. Playwright for Python PyPI version Anaconda version Join Discord. Playwright is a Python library to automate Chromium, Firefox and WebKit browsers with a single API. Playwright delivers automation that is ever-green, capable, reliab

### Creating Custom Tool

In [97]:
from langchain.tools import tool

@tool
def add_numbers(a: int, b:int) -> int:
    "Add two number and return results."
    return  int(a) + int(b)

@tool
def subtract_numbers(a: int, b:int) -> int:
    "Subtract two number and return results."
    return  int(a) - int(b)

@tool
def multiply_numbers(a: int, b:int) -> int:
    "Multiply two number and return results."
    return  int(a) * int(b)

print(add_numbers.invoke({"a": 10, "b": 20}))

30


### Bind tools with LLM

In [104]:
tools = [wikipidia, add_numbers, subtract_numbers, multiply_numbers]

#print(tools)

list_of_tools = {tool.name: tool for tool in tools}

print(list_of_tools)

#respnse = ollama_local_llm.invoke("When did Avatar: Fire and Ash movie released?")
#respnse = ollama_local_llm.invoke("When did Avatar: The Way of Water movie released?")

#print(respnse.content)

llm_with_tools = ollama_local_llm.bind_tools(tools=tools)

tool_call_response = llm_with_tools.invoke("When did Avatar: Fire and Ash movie released?")

print(tool_call_response)

{'wikipedia': WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/home/shashank-sharma/Developer/Projects/langchain-trainings/myenv312/lib/python3.14/site-packages/wikipedia/__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)), 'add_numbers': StructuredTool(name='add_numbers', description='Add two number and return results.', args_schema=<class 'langchain_core.utils.pydantic.add_numbers'>, func=<function add_numbers at 0x7f274cbd2770>), 'subtract_numbers': StructuredTool(name='subtract_numbers', description='Subtract two number and return results.', args_schema=<class 'langchain_core.utils.pydantic.subtract_numbers'>, func=<function subtract_numbers at 0x7f274cbd24b0>), 'multiply_numbers': StructuredTool(name='multiply_numbers', description='Multiply two number and return results.', args_schema=<class 'langchain_core.utils.pydantic.multiply_numbers'>, func=<function multiply_numbers at 0x7f274cbd2820

### Execute the custom tools from LLM

In [108]:
from langchain_core.messages import HumanMessage

# query = "Did Donald Trump won the 2024 presential election?"
query = "What is the sum of 2 and 22?"

message = [HumanMessage(query)]

message.clear()

ai_message = llm_with_tools.invoke(query)

print(ai_message)

message.append(ai_message)

message

content='' additional_kwargs={} response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-05-11T11:18:31.207263795Z', 'done': True, 'done_reason': 'stop', 'total_duration': 280995681, 'load_duration': 89458435, 'prompt_eval_count': 319, 'prompt_eval_duration': 45235489, 'eval_count': 23, 'eval_duration': 134599693, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'} id='lc_run--019e16c2-930d-7aa2-ab40-87b8705564cf-0' tool_calls=[{'name': 'add_numbers', 'args': {'a': '2', 'b': '22'}, 'id': '19866faf-f18f-45bb-a7ca-96ca0c54f14b', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 319, 'output_tokens': 23, 'total_tokens': 342}


[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-05-11T11:18:31.207263795Z', 'done': True, 'done_reason': 'stop', 'total_duration': 280995681, 'load_duration': 89458435, 'prompt_eval_count': 319, 'prompt_eval_duration': 45235489, 'eval_count': 23, 'eval_duration': 134599693, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--019e16c2-930d-7aa2-ab40-87b8705564cf-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': '2', 'b': '22'}, 'id': '19866faf-f18f-45bb-a7ca-96ca0c54f14b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 319, 'output_tokens': 23, 'total_tokens': 342})]

In [109]:
for tool_call in ai_message.tool_calls:
    tool_name = tool_call['name'].lower()

    execute_tool = list_of_tools[tool_name]

    tool_invoke = execute_tool.invoke(tool_call)

    message.append(tool_invoke)

message

[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-05-11T11:18:31.207263795Z', 'done': True, 'done_reason': 'stop', 'total_duration': 280995681, 'load_duration': 89458435, 'prompt_eval_count': 319, 'prompt_eval_duration': 45235489, 'eval_count': 23, 'eval_duration': 134599693, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--019e16c2-930d-7aa2-ab40-87b8705564cf-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': '2', 'b': '22'}, 'id': '19866faf-f18f-45bb-a7ca-96ca0c54f14b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 319, 'output_tokens': 23, 'total_tokens': 342}),
 ToolMessage(content='24', name='add_numbers', tool_call_id='19866faf-f18f-45bb-a7ca-96ca0c54f14b')]

### Invoke all tools from LLM

In [110]:
final_output = llm_with_tools.invoke(message)

print(final_output.content)

The sum of 2 and 22 is 24.
